In [ ]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size, d_model, num_heads, num_layers, output_size):
        super(TimeSeriesTransformer, self).__init__()
        self.d_model = d_model
        
        # Projection de l'entrée vers la dimension du modèle (d_model)
        self.feature_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Couches de l'encodeur Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=num_heads, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Couche de sortie linéaire pour la prédiction
        self.decoder = nn.Linear(d_model, output_size)
        
    def forward(self, x):
        # x: [Batch, Seq_Len, Input_Size]
        x = self.feature_projection(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        # Passage dans le bloc Transformer
        x = self.transformer_encoder(x)
        
        # On utilise généralement la sortie du dernier pas de temps pour prédire la suite
        out = self.decoder(x[:, -1, :])
        return out